# Garment Worker Productivity Prediction
### Exploratory Data Analysis (EDA) and Machine Learning

**Objective:** To explore the garment manufacturing productivity dataset, understand the factors that influence worker productivity, and build machine learning regression models to predict `actual_productivity`.

**Dataset:** `garments_worker_productivity.csv`

## 1. Importing Required Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Loading the Dataset
Keep `garments_worker_productivity.csv` in the same folder as this notebook.

In [ ]:
data = pd.read_csv("garments_worker_productivity.csv")
df = pd.DataFrame(data)

## 3. Displaying the Dataset

In [ ]:
display(df)

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

## 4. Understanding the Dataset

In [ ]:
df.info()

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print("Column names:")
print(df.columns.tolist())

In [ ]:
df.describe(include="all").T

## 5. Data Cleaning

### 5.1 Checking missing values

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum())

### 5.2 Checking duplicate records

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

### 5.3 Removing duplicate records

In [ ]:
df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

### 5.4 Converting the date column

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(df["date"].dtype)

### 5.5 Handling missing values
Numeric columns are filled with their median and categorical columns with their mode. The target column is not imputed; rows without a target value are removed.

In [ ]:
target_column = "actual_productivity"

df = df.dropna(subset=[target_column]).copy()

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

for col in numeric_cols:
    if col != target_column:
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handling:")
display(df.isnull().sum())
print("Updated shape:", df.shape)

## 6. Exploratory Data Analysis (EDA)

### 6.1 Statistical summary

In [ ]:
display(df.describe().T)

### 6.2 Distribution of actual productivity

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="actual_productivity", bins=30, kde=True)
plt.title("Distribution of Actual Worker Productivity")
plt.xlabel("Actual Productivity")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 6.3 Productivity by department

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="department", y="actual_productivity")
plt.title("Actual Productivity by Department")
plt.xlabel("Department")
plt.ylabel("Actual Productivity")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 6.4 Productivity by quarter

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="quarter", y="actual_productivity", errorbar=None)
plt.title("Average Productivity by Quarter")
plt.xlabel("Quarter")
plt.ylabel("Average Actual Productivity")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 6.5 Productivity by day

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="day", y="actual_productivity", errorbar=None)
plt.title("Average Productivity by Day")
plt.xlabel("Day")
plt.ylabel("Average Actual Productivity")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 6.6 Productivity compared with number of workers

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="no_of_workers", y="actual_productivity", hue="department")
plt.title("Number of Workers vs Actual Productivity")
plt.xlabel("Number of Workers")
plt.ylabel("Actual Productivity")
plt.tight_layout()
plt.show()

### 6.7 Productivity compared with incentive

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="incentive", y="actual_productivity")
plt.title("Incentive vs Actual Productivity")
plt.xlabel("Incentive")
plt.ylabel("Actual Productivity")
plt.tight_layout()
plt.show()

### 6.8 Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 7. Outlier Analysis

Boxplots help identify possible outliers. This section visualizes numeric features; it does not automatically remove observations.

In [ ]:
numeric_features_for_boxplot = [
    col for col in df.select_dtypes(include=np.number).columns
    if col != target_column
]

for col in numeric_features_for_boxplot:
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Outlier Analysis - {col}")
    plt.tight_layout()
    plt.show()

## 8. Feature Engineering

Extracting month, day of month, and day of week from the date may help the model capture time-related patterns. The original date column is then removed.

In [ ]:
df["month"] = df["date"].dt.month
df["day_of_month"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

df = df.drop(columns=["date"])

display(df.head())

## 9. Defining Independent and Dependent Variables

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

print("Independent variables (X):", X.shape)
print("Dependent variable (y):", y.shape)
print("\nTarget variable summary:")
display(y.describe())

## 10. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

## 11. Data Preprocessing

Numerical features are imputed using the median and scaled. Categorical features are imputed using the most frequent value and one-hot encoded. Preprocessing is fitted only on the training data through a pipeline to avoid data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

## 12. Building Machine Learning Models

Since `actual_productivity` is a continuous numerical value, regression algorithms are used.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE
    )
}

print("Models selected:")
for name in models:
    print("-", name)

## 13. Model Training and Evaluation

The models are evaluated using MAE, MSE, RMSE, and R-squared. Lower error values are better; a higher R-squared indicates that the model explains more variation in the target.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

trained_models = {}
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    trained_models[name] = pipeline
    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results).sort_values(by="RMSE")
display(results_df.round(4))

## 14. Comparing Model Performance

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.title("Model Comparison - RMSE")
plt.xlabel("Machine Learning Model")
plt.ylabel("Root Mean Squared Error")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="Model", y="R2 Score")
plt.title("Model Comparison - R2 Score")
plt.xlabel("Machine Learning Model")
plt.ylabel("R2 Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 15. Actual vs Predicted Productivity

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_predictions)
min_value = min(y_test.min(), best_predictions.min())
max_value = max(y_test.max(), best_predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title(f"Actual vs Predicted Productivity - {best_model_name}")
plt.xlabel("Actual Productivity")
plt.ylabel("Predicted Productivity")
plt.tight_layout()
plt.show()

print("Best model based on RMSE:", best_model_name)

## 16. Prediction on Sample Test Records

In [ ]:
comparison = pd.DataFrame({
    "Actual Productivity": y_test,
    "Predicted Productivity": best_predictions
})
display(comparison.head(10))

## 17. Final Conclusion

In [ ]:
print("GARMENT WORKER PRODUCTIVITY PREDICTION - SUMMARY")
print("-----------------------------------------------")
print("Dataset rows used:", len(df))
print("Input features:", X.shape[1])
print("Best model based on lowest RMSE:", best_model_name)
print("\nModel performance:")
display(results_df.round(4))

### Conclusion

This project explored the garment worker productivity dataset and examined how variables such as department, quarter, incentives, number of workers, and other recorded production factors relate to actual productivity.

Three regression algorithms were trained and evaluated: Linear Regression, Decision Tree Regressor, and Random Forest Regressor. Their performance was compared using MAE, MSE, RMSE, and R-squared.

The model with the lowest RMSE is selected as the best-performing model in this experiment. The results can help illustrate how machine learning can be applied to estimate worker productivity in garment manufacturing.

**Note:** Model performance should be interpreted using the evaluation results generated when the notebook is run. The analysis shows associations in the dataset and does not prove that any single factor causes productivity to change.